# Train 3D CNN Classifier

This notebook trains a simple 3D CNN on the zebrafish tensor dataset using **time as channels**. To avoid leakage, it loads the persisted base dataset from notebook 5, splits train/validation/holdout on `original_instance_id`, and only then applies random XY rotation augmentation to the training subset.

In [7]:
%load_ext autoreload
%autoreload 2

import warnings

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import classification_report

from zebrafish_ml import (
    Zebrafish3DCNNClassifier,
    augment_training_tensors_with_rotations,
    plot_confusion_matrices,
    plot_training_history,
    split_labeled_tensor_dataset_by_instance,
)
from zebrafish_notebook_utils import configure_full_dataframe_display
from zebrafish_tensor_utils import (
    build_tensor_embedding_2d,
    load_labeled_tensor_dataset,
    plot_tensor_embedding_2d,
)

warnings.filterwarnings("ignore", message="IProgress not found.*")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
configure_full_dataframe_display()

In [9]:
# User inputs
# dataset_artifact_path:
#   relative path under .dataset_cache or an absolute path for the persisted dataset artifact from notebook 5.
#   Dataset composition is read from the saved metadata instead of being repeated in this notebook.
dataset_artifact_path = "moa_4class_t10_z3_y128_x128.pt"
# random_seed:
#   seed used for reproducible split assignment and training-time augmentation angles.
random_seed = 0

# Split and augmentation inputs
holdout_fraction = 0.25
validation_fraction_within_train = 0.2
train_num_random_rotations = 2
rotation_range_degrees = 5.0

# CNN inputs
conv_channels = (16, 32, 64)
kernel_size_z = (1, 1, 1)
kernel_size_xy = (5, 3, 3)
stride_z = (1, 1, 1)
stride_xy = (1, 1, 1)
pool_kernel_z = (1, 1, 1)
pool_kernel_xy = (2, 2, 2)
pool_stride_z = (1, 1, 1)
pool_stride_xy = (2, 2, 2)
embedding_dim = 64
dropout = 0.2

# Optimizer / training inputs
batch_size = 16
epochs = 20
learning_rate = 1e-3
weight_decay = 1e-4
device = None

# Embedding plot inputs
embedding_method = "umap"
umap_n_neighbors = 32
umap_min_dist = 0.1

In [10]:
# Load the base dataset persisted by notebook 5 so we split first and avoid rebuilding tensors here.
dataset = load_labeled_tensor_dataset(dataset_artifact_path)
metadata_all = dataset["metadata"].reset_index(drop=True).copy()

selected_mechanisms = [
    name for label, name in sorted(dataset["label_map"].items()) if int(label) != 0
]
selected_concentrations = sorted(
    metadata_all.loc[~metadata_all["is_control"], "concentration_band"].dropna().unique().tolist()
)
dataset_summary_df = pd.DataFrame(
    [
        {"metric": "dataset_artifact_path", "value": str(dataset_artifact_path)},
        {"metric": "n_examples", "value": int(dataset["tensors"].shape[0])},
        {"metric": "tensor_shape", "value": tuple(dataset["tensors"].shape)},
        {"metric": "n_classes_including_water", "value": int(len(dataset["label_map"]))},
        {"metric": "selected_mechanisms", "value": selected_mechanisms},
        {"metric": "selected_concentrations", "value": selected_concentrations},
        {"metric": "n_original_instances", "value": int(metadata_all["original_instance_id"].nunique()) if "original_instance_id" in metadata_all.columns else None},
    ]
)

display(dataset_summary_df)
display(metadata_all.groupby(["label", "label_name"]).size().reset_index(name="n_examples"))

,metric,value
0,dataset_artifact_path,moa_4class_t10_z3_y128_x128.pt
1,n_examples,837
2,tensor_shape,"(837, 10, 3, 128, 128)"
3,n_classes_including_water,5
4,selected_mechanisms,"[GABAAR_Antagonist, GABAAR_NegativeAllostericModulator, NMDAR_Activation, NMDAR_Antagonist]"
5,selected_concentrations,"[high, mid]"
6,n_original_instances,93


,label,label_name,n_examples
0,0,Water,288
1,1,GABAAR_Antagonist,144
2,2,GABAAR_NegativeAllostericModulator,144
3,3,NMDAR_Activation,117
4,4,NMDAR_Antagonist,144


In [11]:
splits = split_labeled_tensor_dataset_by_instance(
    dataset,
    holdout_fraction=holdout_fraction,
    validation_fraction_within_train=validation_fraction_within_train,
    random_state=random_seed,
)
metadata_all = splits.metadata_all
train_instance_ids = splits.train_instance_ids
val_instance_ids = splits.val_instance_ids
holdout_instance_ids = splits.holdout_instance_ids

X_train_base = splits.X_train_base
y_train_base = splits.y_train_base
metadata_train_base = splits.metadata_train_base

X_val = splits.X_val
y_val = splits.y_val
metadata_val = splits.metadata_val

X_holdout = splits.X_holdout
y_holdout = splits.y_holdout
metadata_holdout = splits.metadata_holdout

X_train, y_train, metadata_train = augment_training_tensors_with_rotations(
    X_train_base,
    y_train_base,
    metadata=metadata_train_base,
    num_random_rotations=train_num_random_rotations,
    rotation_range_degrees=rotation_range_degrees,
    random_state=random_seed,
)

split_summary_df = pd.DataFrame(
    [
        {"split": "train_base", "n_examples": int(len(X_train_base)), "n_original_instances": int(len(train_instance_ids))},
        {"split": "train_augmented", "n_examples": int(len(X_train)), "n_original_instances": int(len(train_instance_ids))},
        {"split": "val", "n_examples": int(len(X_val)), "n_original_instances": int(len(val_instance_ids))},
        {"split": "holdout", "n_examples": int(len(X_holdout)), "n_original_instances": int(len(holdout_instance_ids))},
    ]
)
display(split_summary_df)

,split,n_examples,n_original_instances
0,train_base,495,55
1,train_augmented,1485,55
2,val,126,14
3,holdout,216,24


In [ ]:
model = Zebrafish3DCNNClassifier(
    conv_channels=conv_channels,
    kernel_size_z=kernel_size_z,
    kernel_size_xy=kernel_size_xy,
    stride_z=stride_z,
    stride_xy=stride_xy,
    pool_kernel_z=pool_kernel_z,
    pool_kernel_xy=pool_kernel_xy,
    pool_stride_z=pool_stride_z,
    pool_stride_xy=pool_stride_xy,
    embedding_dim=embedding_dim,
    dropout=dropout,
    batch_size=batch_size,
    epochs=epochs,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    validation_split=0.1,
    random_state=random_seed,
    device=device,
    verbose=True,
)

model.fit(X_train, y_train, validation_data=(X_val, y_val))
model.history_

epoch 001/020 train_loss=1.2673 val_loss=1.2760 eta=11:20
epoch 002/020 train_loss=0.9720 val_loss=1.4161 eta=10:43
epoch 003/020 train_loss=0.8412 val_loss=1.5243 eta=10:08
epoch 004/020 train_loss=0.7064 val_loss=1.5088 eta=09:33
epoch 005/020 train_loss=0.6084 val_loss=1.6552 eta=08:57
epoch 006/020 train_loss=0.5428 val_loss=1.9150 eta=08:21
epoch 007/020 train_loss=0.5164 val_loss=1.7761 eta=07:45
epoch 008/020 train_loss=0.4967 val_loss=1.5720 eta=07:09
epoch 009/020 train_loss=0.4608 val_loss=1.3516 eta=06:33
epoch 010/020 train_loss=0.3841 val_loss=1.0556 eta=05:57
epoch 011/020 train_loss=0.3771 val_loss=2.2760 eta=05:22
epoch 012/020 train_loss=0.3211 val_loss=2.3590 eta=04:46
epoch 013/020 train_loss=0.3033 val_loss=3.5820 eta=04:10


In [ ]:
plot_training_history(model, title="3D CNN train/validation loss");

In [ ]:
holdout_pred = model.predict(X_holdout)
report_df = pd.DataFrame(
    classification_report(
        y_holdout,
        holdout_pred,
        labels=model.classes_,
        target_names=[dataset["label_map"][int(label)] for label in model.classes_],
        output_dict=True,
        zero_division=0,
    )
).T
display(report_df)
plot_confusion_matrices(
    y_holdout,
    holdout_pred,
    class_labels=model.classes_,
    label_map=dataset["label_map"],
);

In [ ]:
holdout_embeddings = torch.from_numpy(model.transform(X_holdout))
holdout_embedding_df = build_tensor_embedding_2d(
    holdout_embeddings,
    y_holdout,
    label_map=dataset["label_map"],
    metadata=metadata_holdout,
    method=embedding_method,
    random_state=random_seed,
    umap_n_neighbors=umap_n_neighbors,
    umap_min_dist=umap_min_dist,
)
display(holdout_embedding_df.head(5))
plot_tensor_embedding_2d(
    holdout_embedding_df,
    title=f"Holdout learned embeddings | {embedding_method.upper()}",
    marker_column="compound",
    show_svm_background=True,
);